# Chapter 3: MNIST From Scratch

Applying everything from notebooks 1 & 2 to a real dataset.

**What we'll do:**
1. Load real image data (MNIST handwritten digits)
2. Explore tensors, shapes, and visualize images
3. Build a neural network from scratch
4. Train it with gradient descent
5. Evaluate accuracy on test data
6. Visualize predictions

This notebook works in Google Colab - no local setup needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Check if GPU is available (Colab offers free GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Loading MNIST

MNIST = 70,000 handwritten digit images (28x28 pixels, grayscale)
- 60,000 for training
- 10,000 for testing

In [ ]:
# Transform: convert images to tensors and normalize to 0-1 range
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL image to tensor, scales to 0-1
])

# Download MNIST (will cache after first download)
train_dataset = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## Part 2: Exploring the Data

Remember: a Dataset returns `(input, target)` tuples.

In [ ]:
# Get one sample
image, label = train_dataset[0]

print(f"Image type: {type(image)}")
print(f"Image shape: {image.shape}")
print(f"Label: {label}")
print(f"\nShape breakdown: (channels, height, width) = {image.shape}")
print(f"  - 1 channel (grayscale, not RGB)")
print(f"  - 28 pixels tall")
print(f"  - 28 pixels wide")
print(f"  - Total pixels: {28 * 28} = 784")

In [ ]:
# Pixel values are normalized to 0-1
print(f"Min pixel value: {image.min():.4f}")
print(f"Max pixel value: {image.max():.4f}")

In [ ]:
# Visualize a single image
plt.figure(figsize=(4, 4))
plt.imshow(image.squeeze(), cmap='gray')  # squeeze removes the channel dim for plotting
plt.title(f"Label: {label}")
plt.axis('off')
plt.show()

In [ ]:
# Visualize a grid of samples
fig, axes = plt.subplots(3, 6, figsize=(12, 6))

for idx, ax in enumerate(axes.flat):
    image, label = train_dataset[idx]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f"{label}")
    ax.axis('off')

plt.suptitle("Sample MNIST Images", fontsize=14)
plt.tight_layout()
plt.show()

## Part 3: DataLoaders (Batching)

DataLoader groups samples into batches for efficient training.

In [ ]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Batch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"\n60,000 / 64 = {60000 / 64:.0f} batches (plus 1 smaller batch for remainder)")

In [ ]:
# Get one batch
images, labels = next(iter(train_loader))

print(f"Batch of images shape: {images.shape}")
print(f"Batch of labels shape: {labels.shape}")
print(f"\nBreakdown: (batch_size, channels, height, width)")

## Part 4: Building the Model

Architecture:
```
Input (784 pixels) → Linear(784→128) → ReLU → Linear(128→64) → ReLU → Linear(64→10)
```

- 784 inputs (28x28 image flattened)
- Two hidden layers (128 and 64 neurons)
- 10 outputs (one per digit 0-9)

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Define layers
        self.flatten = nn.Flatten()  # (batch, 1, 28, 28) → (batch, 784)
        self.layer1 = nn.Linear(784, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 10)
    
    def forward(self, x):
        # Define how data flows through
        x = self.flatten(x)        # Flatten image to 1D
        x = F.relu(self.layer1(x)) # Linear → ReLU
        x = F.relu(self.layer2(x)) # Linear → ReLU
        x = self.layer3(x)         # Final Linear (no activation - done by loss function)
        return x

model = MNISTNet().to(device)
print(model)

In [ ]:
# Count parameters (weights + biases)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"\nBreakdown:")
print(f"  Layer 1: 784 × 128 + 128 = {784*128 + 128:,}")
print(f"  Layer 2: 128 × 64 + 64 = {128*64 + 64:,}")
print(f"  Layer 3: 64 × 10 + 10 = {64*10 + 10:,}")

In [ ]:
# Test forward pass with one batch
images, labels = next(iter(train_loader))
images = images.to(device)

output = model(images)
print(f"Input shape: {images.shape}")
print(f"Output shape: {output.shape}")
print(f"\n64 images → 64 sets of 10 scores (one per digit)")

In [ ]:
# Look at one prediction (before training - random weights)
print(f"Raw scores for first image: {output[0].data}")
print(f"\nPredicted digit: {output[0].argmax().item()}")
print(f"Actual digit: {labels[0].item()}")
print(f"\n(Probably wrong - we haven't trained yet!)")

## Part 5: Loss Function and Optimizer

- **Loss**: CrossEntropyLoss (standard for multi-class classification)
- **Optimizer**: SGD (what we learned about!)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print("Loss function: CrossEntropyLoss")
print("  - Takes raw scores (logits) and targets")
print("  - Applies softmax internally")
print("  - Returns single loss value")
print(f"\nOptimizer: SGD with learning rate = 0.1")

In [ ]:
# Test loss calculation
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

output = model(images)
loss = loss_fn(output, labels)

print(f"Output shape: {output.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Loss: {loss.item():.4f}")
print(f"\n(Random weights → high loss, ~2.3 is typical for 10 classes)")

## Part 6: The Training Loop

Remember the steps:
1. Forward pass (get predictions)
2. Calculate loss
3. Backward pass (compute gradients)
4. Update weights
5. Repeat

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """Train for one complete pass through the data."""
    model.train()  # Set to training mode
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        # Move to device (GPU if available)
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        output = model(images)
        loss = loss_fn(output, labels)
        
        # Backward pass
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute gradients
        optimizer.step()       # Update weights
        
        # Track metrics
        total_loss += loss.item()
        predicted = output.argmax(dim=1)  # Get predicted digit
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
def evaluate(model, loader, loss_fn, device):
    """Evaluate on test data (no gradient updates)."""
    model.eval()  # Set to evaluation mode
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():  # Don't track gradients
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            output = model(images)
            loss = loss_fn(output, labels)
            
            total_loss += loss.item()
            predicted = output.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
# Training!
n_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Training...\n")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>10} | {'Test Loss':>10} | {'Test Acc':>10}")
print("-" * 60)

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f"{epoch+1:>5} | {train_loss:>10.4f} | {train_acc:>10.2%} | {test_loss:>10.4f} | {test_acc:>10.2%}")

print(f"\nFinal test accuracy: {test_acc:.2%}")

In [ ]:
# Plot training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(history['train_loss'], 'b-', label='Train')
ax1.plot(history['test_loss'], 'r-', label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Over Training')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history['train_acc'], 'b-', label='Train')
ax2.plot(history['test_acc'], 'r-', label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Over Training')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 7: Visualizing Predictions

Let's see what the model learned!

In [ ]:
def predict_and_show(model, dataset, indices, device):
    """Show predictions for specific images."""
    model.eval()
    n = len(indices)
    fig, axes = plt.subplots(2, n, figsize=(2*n, 5))
    
    for i, idx in enumerate(indices):
        image, true_label = dataset[idx]
        
        # Get prediction
        with torch.no_grad():
            output = model(image.unsqueeze(0).to(device))  # Add batch dim
            probs = F.softmax(output, dim=1)[0]  # Convert to probabilities
            pred_label = output.argmax(dim=1).item()
        
        # Show image
        axes[0, i].imshow(image.squeeze(), cmap='gray')
        color = 'green' if pred_label == true_label else 'red'
        axes[0, i].set_title(f"True: {true_label}\nPred: {pred_label}", color=color)
        axes[0, i].axis('off')
        
        # Show confidence bars
        axes[1, i].barh(range(10), probs.cpu().numpy())
        axes[1, i].set_yticks(range(10))
        axes[1, i].set_xlim(0, 1)
        axes[1, i].set_xlabel('Confidence')
        if i == 0:
            axes[1, i].set_ylabel('Digit')
    
    plt.tight_layout()
    plt.show()

# Show some test predictions
predict_and_show(model, test_dataset, [0, 1, 2, 3, 4], device)

In [ ]:
# Find some mistakes
def find_mistakes(model, dataset, device, n=5):
    """Find misclassified images."""
    model.eval()
    mistakes = []
    
    for idx in range(len(dataset)):
        image, true_label = dataset[idx]
        
        with torch.no_grad():
            output = model(image.unsqueeze(0).to(device))
            pred_label = output.argmax(dim=1).item()
        
        if pred_label != true_label:
            mistakes.append(idx)
            if len(mistakes) >= n:
                break
    
    return mistakes

mistakes = find_mistakes(model, test_dataset, device, n=5)
print(f"Found {len(mistakes)} mistakes. Showing them:")
predict_and_show(model, test_dataset, mistakes, device)

## Part 8: Understanding What the Model Learned

Let's peek at the weights of the first layer.

In [ ]:
# Get first layer weights
weights = model.layer1.weight.data.cpu()
print(f"First layer weights shape: {weights.shape}")
print(f"  - 128 neurons, each with 784 weights (one per pixel)")

In [ ]:
# Visualize some neurons' weights as 28x28 images
fig, axes = plt.subplots(4, 8, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    # Reshape weights to image shape
    neuron_weights = weights[i].reshape(28, 28)
    ax.imshow(neuron_weights, cmap='RdBu', vmin=-0.5, vmax=0.5)
    ax.axis('off')
    ax.set_title(f'Neuron {i}')

plt.suptitle('First Layer Weights (what each neuron "looks for")', fontsize=14)
plt.tight_layout()
plt.show()

print("Red = positive weight (looks for ink here)")
print("Blue = negative weight (looks for NO ink here)")

## Part 9: Experimenting

Try changing these and see what happens!

In [ ]:
# Experiment: Different learning rates
def quick_train(lr, epochs=3):
    """Train a fresh model with given learning rate."""
    model = MNISTNet().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    
    losses = []
    for epoch in range(epochs):
        train_loss, _ = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        losses.append(train_loss)
    
    _, test_acc = evaluate(model, test_loader, loss_fn, device)
    return losses, test_acc

# Compare learning rates
learning_rates = [0.001, 0.01, 0.1, 1.0]
results = {}

print("Testing different learning rates...\n")
for lr in learning_rates:
    losses, acc = quick_train(lr)
    results[lr] = (losses, acc)
    print(f"LR = {lr}: Final accuracy = {acc:.2%}")

In [ ]:
# Plot learning rate comparison
plt.figure(figsize=(10, 5))

for lr, (losses, acc) in results.items():
    plt.plot(losses, label=f'LR={lr} (acc={acc:.1%})', marker='o')

plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Effect of Learning Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Part 10: Simplest Possible Model (Linear Only)

What if we use NO hidden layers? Just input → output?

In [ ]:
# Simplest model: just one linear layer
simple_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 10)  # Direct: pixels → digit scores
).to(device)

simple_optimizer = torch.optim.SGD(simple_model.parameters(), lr=0.1)

print("Simple model (no hidden layers):")
print(simple_model)
print(f"\nParameters: {sum(p.numel() for p in simple_model.parameters()):,}")
print("(Much fewer than our 109K parameter model!)")

In [ ]:
# Train the simple model
print("Training simple model...\n")

for epoch in range(5):
    train_loss, train_acc = train_one_epoch(simple_model, train_loader, loss_fn, simple_optimizer, device)
    test_loss, test_acc = evaluate(simple_model, test_loader, loss_fn, device)
    print(f"Epoch {epoch+1}: Train acc = {train_acc:.2%}, Test acc = {test_acc:.2%}")

print(f"\nSimple model final: {test_acc:.2%}")
print(f"Deep model final:   {history['test_acc'][-1]:.2%}")
print(f"\nThe hidden layers help!")

In [ ]:
# Visualize what the simple model learned
# Each of the 10 output neurons has 784 weights - one per pixel
simple_weights = simple_model[1].weight.data.cpu()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    weights_img = simple_weights[i].reshape(28, 28)
    ax.imshow(weights_img, cmap='RdBu')
    ax.set_title(f'Digit {i}')
    ax.axis('off')

plt.suptitle('Simple Model Weights: What each digit "looks for"', fontsize=14)
plt.tight_layout()
plt.show()

print("You can see templates of each digit!")
print("Red = positive (expects ink), Blue = negative (expects no ink)")

## Key Takeaways

1. **MNIST images** are tensors of shape `(1, 28, 28)` = 784 pixels

2. **DataLoader** batches data for efficient training: `(64, 1, 28, 28)`

3. **The training loop**:
   - Forward pass → Loss → Backward pass → Update weights
   - Repeat for each batch, for multiple epochs

4. **More layers = more capacity** to learn complex patterns

5. **Learning rate matters** - too small is slow, too big overshoots

6. **Watch train vs test** - gap indicates overfitting

7. **Weights are interpretable** - especially in simple models, you can see what patterns they detect

## Exercises

Try these to deepen understanding:

1. **Change architecture**: Add another hidden layer. Does accuracy improve?

2. **Change hidden sizes**: Try 256→128→64 instead of 128→64. What happens?

3. **Train longer**: Run for 20 epochs. Do train and test accuracy diverge? (overfitting)

4. **Different optimizer**: Replace SGD with `torch.optim.Adam(model.parameters(), lr=0.001)`

5. **Binary classification**: Modify to only classify 3 vs 7 (like the book)